# 00 - Introduction to SAPIENT in Python

## Learning Objectives

By the end of this notebook, you will:

1. Understand what SAPIENT is and why it matters in defence systems
2. Understand the purpose and structure of this educational repository
3. Set up your development environment for the course
4. Verify your Python installation and dependencies
5. Understand the learning path ahead

---

## What is SAPIENT?

**SAPIENT** (Sensors And Processing for INTelligence) is a standardized protocol for connecting sensor systems in multi-sensor networks.

### Why Does It Exist?

Modern defence and security operations rely on multiple sensor types:
- **Radars** (ground, air, maritime)
- **Cameras** (EO/IR)
- **Acoustic sensors**
- **RF detectors**
- **LIDAR systems**

Historically, each sensor had its own proprietary protocol. Integrating multiple sensors required:
- Custom adapters for each sensor type
- Different message formats
- Vendor lock-in
- High integration costs

**SAPIENT solves this** by providing a common "language" for sensors to communicate.

### The Core Idea

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│    Radar     │     │   Camera     │     │   Acoustic   │
│              │     │              │     │              │
└──────┬───────┘     └──────┬───────┘     └──────┬───────┘
       │                    │                    │
       │ SAPIENT Messages   │                    │
       └────────────────────┼────────────────────┘
                            │
                    ┌───────▼────────┐
                    │  Fusion Node   │
                    │  (Combines     │
                    │   sensor data) │
                    └───────┬────────┘
                            │
                    ┌───────▼────────┐
                    │   Command &    │
                    │   Control      │
                    └────────────────┘
```

All sensors speak SAPIENT → Easy integration → Interoperability

## About This Repository

### What This Is

This is an **educational implementation** designed to teach you SAPIENT concepts through hands-on coding.

You will build:
- A simulated radar sensor
- A sensor node that generates SAPIENT messages
- A fusion node that combines data from multiple sensors
- A simple command and control display

### What This Is NOT

⚠️ **This is NOT a complete implementation of the official SAPIENT standard.**

Where the official specification is not publicly available or is complex, we will:
- Create simplified educational versions
- Clearly mark them as "educational simplifications"
- Focus on teaching the concepts rather than implementing every detail

### Learning Path

```
00. Introduction (you are here)
01. What is SAPIENT?
02. System Architecture
03. Protocol Buffers
04. First Message
05. Building a Sensor Node
06. Simulating a Radar
07. Track Generation
08. Building a Fusion Node
09. Command and Control
10. End-to-End Demo
```

Each notebook builds on the previous ones, ending with a complete working system.

## Prerequisites

This course assumes you are familiar with:

✅ **Python** (classes, type hints, async/await)  
✅ **Linux** (command line, networking basics)  
✅ **Networking** (TCP/UDP, sockets)  
✅ **Radar fundamentals** (range, azimuth, Doppler)  
✅ **Software-Defined Radios (SDRs)** (at a conceptual level)  

You do NOT need prior experience with:
- SAPIENT
- Protocol Buffers
- Sensor fusion

We will teach these from first principles.

## Environment Setup

### Python Version

This course requires **Python 3.12 or later**.

Let's verify your Python version:

In [1]:
import sys
from typing import Tuple

def check_python_version() -> Tuple[bool, str]:
    """
    Check if Python version meets requirements.
    
    Returns:
        Tuple of (meets_requirements, version_string)
    """
    version = sys.version_info
    version_str = f"{version.major}.{version.minor}.{version.micro}"
    meets_requirements = version.major == 3 and version.minor >= 12
    
    return meets_requirements, version_str

# Check version
meets_reqs, version = check_python_version()

if meets_reqs:
    print(f"✅ Python {version} - Ready to go!")
else:
    print(f"❌ Python {version} - Please upgrade to Python 3.12 or later")
    print("\nInstallation guide: https://www.python.org/downloads/")

✅ Python 3.12.3 - Ready to go!


### Required Dependencies

Throughout this course, we will use:

- **protobuf** - For message serialization
- **asyncio** - For asynchronous networking (Python standard library)
- **matplotlib** - For visualization
- **numpy** - For numerical computation
- **dataclasses** - For clean data structures (Python standard library)

Let's verify these are available:

In [2]:
import importlib
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class DependencyCheck:
    """Result of a dependency availability check."""
    name: str
    available: bool
    version: Optional[str] = None
    note: str = ""

def check_dependencies() -> List[DependencyCheck]:
    """
    Check availability of required dependencies.
    
    Returns:
        List of dependency check results
    """
    dependencies = [
        ("asyncio", "Python standard library"),
        ("dataclasses", "Python standard library"),
        ("logging", "Python standard library"),
        ("socket", "Python standard library"),
    ]
    
    external_dependencies = [
        "google.protobuf",
        "matplotlib",
        "numpy",
    ]
    
    results = []
    
    # Check standard library modules
    for module_name, note in dependencies:
        try:
            importlib.import_module(module_name)
            results.append(DependencyCheck(module_name, True, note=note))
        except ImportError:
            results.append(DependencyCheck(module_name, False, note=note))
    
    # Check external dependencies
    for module_name in external_dependencies:
        try:
            module = importlib.import_module(module_name)
            version = getattr(module, "__version__", "unknown")
            results.append(DependencyCheck(module_name, True, version=version))
        except ImportError:
            results.append(DependencyCheck(module_name, False))
    
    return results

# Run dependency check
results = check_dependencies()

print("Dependency Check Results:")
print("=" * 60)

all_available = True
for result in results:
    status = "✅" if result.available else "❌"
    version_info = f" (v{result.version})" if result.version else ""
    note_info = f" - {result.note}" if result.note else ""
    print(f"{status} {result.name}{version_info}{note_info}")
    
    if not result.available:
        all_available = False

print("=" * 60)

if all_available:
    print("\n🎉 All dependencies are available!")
else:
    print("\n⚠️  Some dependencies are missing. Install them with:")
    print("\npip install protobuf matplotlib numpy")

Dependency Check Results:
✅ asyncio - Python standard library
✅ dataclasses - Python standard library
✅ logging - Python standard library
✅ socket - Python standard library
✅ google.protobuf (v7.35.1)
✅ matplotlib (v3.11.1)
✅ numpy (v2.5.1)

🎉 All dependencies are available!


### Installing Missing Dependencies

If any external dependencies are missing, install them:

```bash
pip install protobuf matplotlib numpy
```

Or use the requirements file (we'll create this in the next cell):

In [ ]:
# Create a requirements.txt file for easy installation
requirements = """
# Core dependencies for sapient-python educational repository
protobuf>=5.27.0
numpy>=1.26.0
matplotlib>=3.8.0

# Development and notebook support
jupyter>=1.0.0
ipython>=8.12.0
""".strip()

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ Created requirements.txt")
print("\nInstall with: pip install -r requirements.txt")

## Your First SAPIENT Concept: Messages

At its core, SAPIENT is about **messages**.

Sensors send messages like:
- "I detected something at range 1500m, azimuth 45°"
- "Here's my current status: operational, 85% battery"
- "I acknowledge your task assignment"

Let's create a simple educational message structure to illustrate the concept:

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Literal
from uuid import uuid4

@dataclass
class SimpleMessage:
    """
    A simplified educational message structure.
    
    Note: This is NOT an official SAPIENT message format.
    This is a teaching tool to introduce the concept of structured messages.
    
    Attributes:
        message_id: Unique identifier for this message
        timestamp: When the message was created (UTC)
        sender_id: Who sent this message
        message_type: Type of message being sent
        content: The actual message content
    """
    message_id: str = field(default_factory=lambda: str(uuid4()))
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    sender_id: str = "unknown"
    message_type: Literal["detection", "status", "command", "ack"] = "status"
    content: dict = field(default_factory=dict)
    
    def __str__(self) -> str:
        """Human-readable representation of the message."""
        time_str = self.timestamp.strftime("%Y-%m-%d %H:%M:%S UTC")
        return (
            f"Message[{self.message_type.upper()}]\n"
            f"  ID: {self.message_id}\n"
            f"  From: {self.sender_id}\n"
            f"  Time: {time_str}\n"
            f"  Content: {self.content}"
        )

# Create and display a sample message
message = SimpleMessage(
    sender_id="radar-001",
    message_type="detection",
    content={
        "range_m": 1500.0,
        "azimuth_deg": 45.0,
        "confidence": 0.92
    }
)

print("Example SAPIENT-style Message:")
print("=" * 60)
print(message)
print("=" * 60)
print("\n✅ This demonstrates the concept of structured messages.")
print("   In later notebooks, we'll use proper Protocol Buffers.")

Example SAPIENT-style Message:
Message[DETECTION]
  ID: 63166693-b182-44b6-9508-c09f31e50ad4
  From: radar-001
  Time: 2026-10-28 09:34:24 UTC
  Content: {'range_m': 1500.0, 'azimuth_deg': 45.0, 'confidence': 0.92}

✅ This demonstrates the concept of structured messages.
   In later notebooks, we'll use proper Protocol Buffers.


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/derrington/repo/sapient/.venv3.12/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/derrington/repo/sapient/.venv3.12/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/home/derrington/repo/sapient/.venv3.12/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/derrington/repo/sapient/.venv3.12/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/derrington/repo/sapien

## Key Takeaways

From this introduction, you should understand:

1. **SAPIENT is a protocol** for standardizing sensor communication
2. **This repository is educational** - we build simplified versions to teach concepts
3. **Messages are central** - sensors communicate via structured messages
4. **Your environment is ready** - Python 3.12+ with required dependencies

### What's Next?

In **01_What_is_SAPIENT.ipynb**, we'll dive deeper into:
- The history and motivation behind SAPIENT
- Key message types (Detection, Status, Task, etc.)
- The SAPIENT architecture
- How sensors interact with fusion nodes

## Exercises

### Exercise 1: Modify the Message

Create a `SimpleMessage` representing a camera detecting a person:
- Sender: "camera-003"
- Message type: "detection"
- Content: object_class="person", confidence=0.88, pixel_location=(512, 384)

In [ ]:
# Your solution here


### Exercise 2: Message Factory

Create a helper function `create_status_message(sender_id: str, battery_percent: float)` that returns a status message with battery information.

In [ ]:
# Your solution here


### Exercise 3: Think About Scale

Consider a scenario with:
- 10 radar sensors
- 5 cameras
- Each sensor sends 1 detection message per second

Questions:
1. How many messages per minute?
2. What challenges might arise at this scale?
3. Why might we need structured binary formats instead of JSON?

In [ ]:
# Calculate and discuss
num_radars = 10
num_cameras = 5
messages_per_second = 1

# Your calculations here


## Extension Ideas

If you want to explore further before the next notebook:

1. **Add validation** to `SimpleMessage` - check that ranges are positive, timestamps are not in the future
2. **Serialize to JSON** - add a method to convert the message to JSON for debugging
3. **Message history** - create a simple in-memory list to store messages and query by sender or type
4. **Research** - Look up "sensor fusion" and "track correlation" to get ahead on concepts we'll cover

---

**Next:** [01_What_is_SAPIENT.ipynb](./01_What_is_SAPIENT.ipynb)